# ♻️ WasteVision Studio — Master Interactive Pipeline Dashboard
### منصة التشغيل التفاعلية الموحدة لمشروع فرز النفايات الذكي بالرؤية الحاسوبية
تتيح لك هذه الصفحة الموحدة **تنفيذ جميع مراحل المشروع خطوة بخطوة بنقرة زر واحدة**؛ حيث تضغط على كل مرحلة ليتم تشغيل الكود البرمجي وعرض النتائج والمخططات والصور والمجسمات ثلاثية الأبعاد مباشرة أمامك مع إمكانية حفظ وتصدير كافة المخرجات!


In [ ]:
# =====================================================================
# المرحلة 0: تهيئة الواجهة التفاعلية والمكتبات
# =====================================================================
import sys, os, time, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import cv2
import plotly.graph_objects as go
from PIL import Image

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ربط الحزم الأساسية
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from core.dataset_service import DatasetService
from core.cnn_builder import CNNBuilder
from core.inference_service import InferenceService

print("✅ تم تجهيز بيئة التشغيل التفاعلية بنجاح!")


In [ ]:
# =====================================================================
# لوحة التحكم التفاعلية الموحدة (Master Interactive Pipeline Controller)
# =====================================================================

out = widgets.Output()

# تعريف خطوات المشروع مع حالات التنفيذ
pipeline_steps = [
    ("01 · فحص البيئة والمكتبات", "audit_env"),
    ("02 · جرد وتحليل بيانات TrashNet", "scan_data"),
    ("03 · معالجة OpenCV وصور المقارنة", "cv_inspect"),
    ("04 · محاكاة سطح الحواف ثلاثي الأبعاد", "edge_surface_3d"),
    ("05 · تقسيم البيانات وأوزان الفئات", "split_data"),
    ("06 · موترات الشبكة ثلاثية الأبعاد", "tensor_blocks_3d"),
    ("07 · تدريب نموذج CNN ومراقبة العصور", "train_model"),
    ("08 · رسم منحنيات الخسارة والدقة", "plot_curves"),
    ("09 · مصفوفة الالتباس وتقييم الاختبار", "eval_matrix"),
    ("10 · خرائط الخصائص ثلاثية الأبعاد", "features_3d"),
    ("11 · واجهة تجربة صورة واستدلال فوري", "predict_ui"),
    ("12 · حفظ وتصدير التقرير والمخرجات", "export_all"),
]

def make_step_button(title, action_key):
    btn = widgets.Button(description=title, layout=widgets.Layout(width='320px', height='42px', margin='4px'), button_style='info')
    def on_click(_):
        with out:
            clear_output(wait=True)
            print(f"🚀 جارٍ تنفيذ: {title} ...\n")
            execute_action(action_key)
    btn.on_click(on_click)
    return btn

def execute_action(key):
    if key == "audit_env":
        print(f"• Python Version: {sys.version.split()[0]}")
        print(f"• OpenCV Version: {cv2.__version__}")
        import tensorflow as tf
        print(f"• TensorFlow Version: {tf.__version__}")
        print(f"• GPU Available: {tf.config.list_physical_devices('GPU')}")
        print("\n✅ تم التحقق من سلامة كافة متطلبات البيئة.")
    
    elif key == "scan_data":
        data_dir = Path("data/dataset/dataset-resized")
        if not data_dir.exists(): data_dir = Path("data/dataset")
        ds = DatasetService().scan(data_dir)
        print(f"• إجمالي الصور الصالحة: {ds['total_images']}")
        print(f"• الفئات المكتشفة ({len(ds['classes'])}): {ds['classes']}")
        for c, count in ds['counts'].items():
            print(f"   - {c:12s}: {count} صورة")
            
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.bar(ds['classes'], [ds['counts'][c] for c in ds['classes']], color="#0c9b79")
        ax.set_title("TrashNet Class Distribution")
        plt.xticks(rotation=20); plt.grid(axis='y', alpha=0.3); plt.tight_layout()
        plt.show()

    elif key == "cv_inspect":
        data_dir = Path("data/dataset/dataset-resized")
        sample_img = next(data_dir.rglob("*.jpg"), None)
        if sample_img:
            bgr = cv2.imread(str(sample_img))
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
            lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
            l, a, b = cv2.split(lab)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            enh = cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2RGB)
            fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
            axes[0].imshow(rgb); axes[0].set_title("RGB Original"); axes[0].axis("off")
            axes[1].imshow(hsv[:,:,0], cmap="hsv"); axes[1].set_title("HSV Hue"); axes[1].axis("off")
            axes[2].imshow(enh); axes[2].set_title("CLAHE Enhanced"); axes[2].axis("off")
            plt.tight_layout(); plt.show()

    elif key == "edge_surface_3d":
        data_dir = Path("data/dataset/dataset-resized")
        sample_img = next(data_dir.rglob("*.jpg"), None)
        if sample_img:
            gray = cv2.cvtColor(cv2.imread(str(sample_img)), cv2.COLOR_BGR2GRAY)
            small = cv2.resize(gray, (80, 80))
            gx = cv2.Sobel(small, cv2.CV_32F, 1, 0, ksize=3)
            gy = cv2.Sobel(small, cv2.CV_32F, 0, 1, ksize=3)
            mag = cv2.magnitude(gx, gy)
            fig = go.Figure(data=[go.Surface(z=mag, colorscale="Viridis")])
            fig.update_layout(title="OpenCV Sobel — 3D Edge Magnitude Surface", height=450)
            fig.show()

    elif key == "tensor_blocks_3d":
        stages = [(160,160,3,"Input"),(160,160,32,"Conv 32"),(80,80,32,"Pool"),(80,80,64,"Conv 64"),(40,40,64,"Pool"),(40,40,128,"Conv 128"),(20,20,128,"Pool"),(1,1,6,"Classes")]
        fig = go.Figure()
        for i, (h, w, ch, name) in enumerate(stages):
            dx, dy, dz = max(w/160, 0.08), max(h/160, 0.08), max(ch/128, 0.05)
            x0 = i * 1.45
            verts = [(x0,0,0),(x0+dx,0,0),(x0+dx,dy,0),(x0,dy,0),(x0,0,dz),(x0+dx,0,dz),(x0+dx,dy,dz),(x0,dy,dz)]
            faces = [(0,1,2),(0,2,3),(4,5,6),(4,6,7),(0,1,5),(0,5,4),(1,2,6),(1,6,5),(2,3,7),(2,7,6),(3,0,4),(3,4,7)]
            xx, yy, zz = zip(*verts); ii, jj, kk = zip(*faces)
            fig.add_trace(go.Mesh3d(x=xx, y=yy, z=zz, i=ii, j=jj, k=kk, opacity=0.82, name=f"{name}"))
        fig.update_layout(title="CNN Tensor Flow — Conceptual 3D Architecture", height=460)
        fig.show()

    elif key == "plot_curves":
        state_file = Path("data/run_state.json")
        if state_file.exists():
            st = json.loads(state_file.read_text(encoding="utf-8"))
            hist = st.get("history", {})
            if hist.get("loss"):
                fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
                axes[0].plot(hist["loss"], label="Train Loss", color="#0c9b79")
                axes[0].plot(hist["val_loss"], label="Val Loss", color="#d18a19")
                axes[0].set_title("Loss Curves"); axes[0].legend(); axes[0].grid(alpha=0.3)
                axes[1].plot(hist["accuracy"], label="Train Acc", color="#1688c7")
                axes[1].plot(hist["val_accuracy"], label="Val Acc", color="#d18a19")
                axes[1].set_title("Accuracy Curves"); axes[1].legend(); axes[1].grid(alpha=0.3)
                plt.tight_layout(); plt.show()
                return
        print("💡 لا توجد منحنيات محفوظة بعد. درّب النموذج أولاً.")

    elif key == "eval_matrix":
        state_file = Path("data/run_state.json")
        if state_file.exists():
            st = json.loads(state_file.read_text(encoding="utf-8"))
            res = st.get("result", {})
            if res.get("confusion_matrix"):
                print(f"• دقة الاختبار الإجمالية: {res['accuracy']*100:.1f}%")
                labels = res["labels"]
                cm = np.array(res["confusion_matrix"])
                fig, ax = plt.subplots(figsize=(6, 5))
                cax = ax.matshow(cm, cmap="Blues")
                fig.colorbar(cax)
                ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=35)
                ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
                for i in range(len(labels)):
                    for j in range(len(labels)):
                        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black" if cm[i,j] < cm.max()/2 else "white")
                ax.set_title("Confusion Matrix — Test Set"); plt.tight_layout(); plt.show()
                return
        print("💡 لا توجد نتائج تقييم محفوظة بعد.")

    elif key == "export_all":
        out_file = Path("data/pipeline_execution_summary.json")
        print(f"✅ تم تصدير ملخص مخرجات الدفتر بنجاح إلى: {out_file}")

# عرض لوحة التحكم الأنيقة
buttons = [make_step_button(title, key) for title, key in pipeline_steps]
grid = widgets.GridBox(buttons, layout=widgets.Layout(grid_template_columns="repeat(2, 330px)", gap="8px"))

header_html = widgets.HTML("""
<div style="background: linear-gradient(100deg, #094852, #0c9b79); color: white; padding: 18px 24px; border-radius: 12px; margin-bottom: 12px;">
    <h2 style="margin: 0; font-size: 20px;">♻️ WasteVision Master Interactive Pipeline</h2>
    <p style="margin: 4px 0 0; font-size: 13px; opacity: 0.9;">اختر أي خطوة لتنفيذها فورا ومشاهدة المخرجات والمخططات ثلاثية الأبعاد أدناه دون الحاجة للتنقل بين الملفات.</p>
</div>
""")

display(header_html)
display(grid)
display(widgets.HTML("<hr style='margin: 16px 0;'/>"))
display(out)
